# Step 3: Feature Selection

1. use selected feature, fit xgboost

2. Optuna to tune

3. cross validation

4. final model; score test set



In [39]:
import os
import numpy as np
import pandas as pd

import optuna

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import f1_score, make_scorer, accuracy_score

import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier


In [3]:
current_dir = os.getcwd()
home_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
data_dir = os.path.join(home_dir, 'data/')

In [4]:
train_cat = pd.read_csv(os.path.join(data_dir, 'intermediate/encoded_train_cat.csv'))
train_quant = pd.read_csv(os.path.join(data_dir, 'intermediate/train_quant_filled.csv'))
train_fcm_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/train_pca_100.csv'))
train_fcm_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/train_pca_500.csv'))

test_cat = pd.read_csv(os.path.join(data_dir, 'intermediate/encoded_test_cat.csv'))
test_quant = pd.read_csv(os.path.join(data_dir, 'intermediate/test_quant_filled.csv'))
test_fcm_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/test_pca_100.csv'))
test_fcm_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/test_pca_500.csv'))

# selected features
features_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/common_features_selected.csv'), header=None)
features_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/common_features_selected_500.csv'), header=None)

train_label = pd.read_excel(os.path.join(data_dir, 'TRAIN_NEW/TRAINING_SOLUTIONS.xlsx'))

In [17]:
# Join data
X = train_cat.merge(train_quant, on='participant_id', how='left').merge(train_fcm_100, on='participant_id', how='left')
# X = X.drop(columns=['participant_id'], axis=1)
y = train_label.copy()
# y = y.drop(columns=['participant_id'], axis=1)

In [18]:
features_100_list = features_100[0].tolist()
features_100_list = [x.removesuffix('_scaled') for x in features_100_list]

In [19]:
# only keep the selected features
X = X[features_100_list + ['participant_id']]

In [20]:
# Split training and test data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

In [21]:
X_train = X_train.drop(columns=['participant_id'], axis=1)
y_train = y_train.drop(columns=['participant_id'], axis=1)

In [9]:
# Custom scoring function for weighted F1 score
def weighted_f1(y_true, y_pred):
    # Ensure that y_true and y_pred are numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    f1_adhd = f1_score(y_true[:, 0], y_pred[:, 0])
    f1_sex = f1_score(y_true[:, 1], y_pred[:, 1])
    return (2/3) * f1_adhd + (1/3) * f1_sex

In [10]:
# Objective function for Optuna optimization
def objective(trial):
    # Define the hyperparameters to tune
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'n_jobs': -1
    }

    # Define the base model
    base_model = xgb.XGBClassifier(**params)

    # Wrap it with MultiOutputClassifier
    multi_model = MultiOutputClassifier(base_model)

    # Perform cross-validation (here we're using 5-fold cross-validation)
    scores = cross_val_score(multi_model, X_train, y_train, cv=5, scoring=make_scorer(weighted_f1))

    # Return the mean of the cross-validation scores
    return np.mean(scores)


In [11]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best params:", study.best_trial.params)

[I 2025-04-27 13:23:55,749] A new study created in memory with name: no-name-34ffed28-e230-433a-b494-9a7eb932e7a6
[I 2025-04-27 13:23:58,630] Trial 0 finished with value: 0.57006034710544 and parameters: {'max_depth': 7, 'learning_rate': 0.15661206095966232, 'n_estimators': 205, 'subsample': 0.6437351985429576, 'colsample_bytree': 0.9778598292119298, 'gamma': 0.7988460772800354}. Best is trial 0 with value: 0.57006034710544.
[I 2025-04-27 13:24:00,917] Trial 1 finished with value: 0.567299180031205 and parameters: {'max_depth': 3, 'learning_rate': 0.29997927689711557, 'n_estimators': 295, 'subsample': 0.7114515815147446, 'colsample_bytree': 0.8499071621915358, 'gamma': 0.44150766581539047}. Best is trial 0 with value: 0.57006034710544.
[I 2025-04-27 13:24:02,754] Trial 2 finished with value: 0.5875856294994438 and parameters: {'max_depth': 8, 'learning_rate': 0.1995197500884828, 'n_estimators': 205, 'subsample': 0.5428582152555128, 'colsample_bytree': 0.6096537322217378, 'gamma': 2.876

Best params: {'max_depth': 8, 'learning_rate': 0.21037904448578923, 'n_estimators': 187, 'subsample': 0.5059509458275161, 'colsample_bytree': 0.7347434800064073, 'gamma': 4.929441168440217}


In [12]:
params = study.best_trial.params
params 


{'max_depth': 8,
 'learning_rate': 0.21037904448578923,
 'n_estimators': 187,
 'subsample': 0.5059509458275161,
 'colsample_bytree': 0.7347434800064073,
 'gamma': 4.929441168440217}

In [14]:
# train with the best params
best_model = xgb.XGBClassifier(**params)
best_multi_model = MultiOutputClassifier(best_model)
best_multi_model.fit(X_train, y_train)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.7347434800064073,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              gamma=4.929441168440217,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.21037904448578923,
                                              max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=8,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=187, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...))

In [23]:
participant_id = X_val['participant_id']
X_val = X_val.drop(columns = 'participant_id')

In [24]:
# score test set
y_pred = best_multi_model.predict(X_val)

In [31]:
y

,participant_id,ADHD_Outcome,Sex_F
0,UmrK0vMLopoR,1,1
1,CPaeQkhcjg7d,1,0
2,Nb4EetVPm3gs,1,0
3,p4vPhVu91o4b,1,1
4,M09PXs7arQ5E,1,1
...,...,...,...
1208,Atx7oub96GXS,0,0
1209,groSbUfkQngM,0,1
1210,zmxGvIrOD0bt,0,1
1211,rOmWFuJCud5G,0,0


In [32]:
predictions_df = pd.DataFrame(
    y_pred,
    columns=['Predicted_ADHD', 'Predicted_Gender']
)

# Combine participant IDs with predictions
result_df = pd.concat([participant_id.reset_index(drop=True), predictions_df], axis=1)

# Print or save the DataFrame
print(result_df)

    participant_id  Predicted_ADHD  Predicted_Gender
0     ExV0vwCIzAWp               0                 0
1     oSlcxZbncT7a               1                 0
2     28mwHUApaS7q               1                 0
3     8E7WIqYsBQBj               1                 1
4     Hn7obzzz4omm               1                 0
..             ...             ...               ...
359   0FUWCjn9YMN1               1                 0
360   d3VfkRYzJJXX               1                 0
361   vRp3pYC1P44F               1                 0
362   ePfFpufOYJFy               0                 1
363   Jz96BoYNDD51               1                 0

[364 rows x 3 columns]


In [34]:
y_val = y_val.drop(columns = 'participant_id')

In [35]:
f1_test = weighted_f1(y_val, predictions_df)
print(f"Weighted F1 score on test set: {f1_test}")

Weighted F1 score on test set: 0.6047352572305302


In [36]:
def multi_output_accuracy(y_true, y_pred):
    # Ensure y_true and y_pred are NumPy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Compute accuracy for each target variable and return the mean
    return np.mean([accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])])

In [40]:
accuracy_test = multi_output_accuracy(y_val, predictions_df)
print(f"Multi-output accuracy on test set: {accuracy_test}")

Multi-output accuracy on test set: 0.635989010989011
